In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from suite2p.extraction import dcnv

/home/zhangl5@hhmi.org/Desktop/zhangl5/Data-Format/manual/majnik2025/.venv/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [2]:
# list the path of all sessions for subject 'jm039'

base_path = './data'
subject = 'jm039'
# now get subdirectories (of 'jm039/')
all_session_dir = [f.path for f in os.scandir(os.path.join(base_path, subject)) if f.is_dir()]
all_session_dir.sort()  # sort them to be in chronological order

print(all_session_dir)

['./data/jm039/2024-04-30_a', './data/jm039/2024-05-01_a', './data/jm039/2024-05-02_a', './data/jm039/2024-05-03_a', './data/jm039/2024-05-04_a', './data/jm039/2024-05-05_a', './data/jm039/2024-05-06_a']


### Neural Recording

In [3]:
# raw F data
for session in all_session_dir:
    F = np.load(os.path.join(session, 'suite2p', 'plane0', 'F.npy'))
    print('shape', F.shape, 'min:', '%.2f' % F.min(), 'max:', '%.2f' % F.max())

shape (746, 54000) min: 4.47 max: 2518.61
shape (746, 54000) min: 13.89 max: 2403.75
shape (746, 54000) min: 12.71 max: 2914.94
shape (746, 54000) min: 7.94 max: 3114.91
shape (746, 54000) min: 14.37 max: 3234.88
shape (746, 54000) min: 12.83 max: 3979.76
shape (746, 54000) min: 12.64 max: 4004.24


In [4]:
F = np.load(os.path.join(all_session_dir[0], 'suite2p', 'plane0', 'F.npy'))
Fneu = np.load(os.path.join(all_session_dir[0], 'suite2p', 'plane0', 'Fneu.npy'))
print(Fneu.shape, Fneu.min(), Fneu.max())

(746, 54000) 18.856354 490.19037


In [5]:
# test preprocessing for one session
# https://suite2p.readthedocs.io/en/latest/deconvolution/
batch_size = 128
neucoeff = 0.7
fs = 30 # 30Hz sampling rate

Fc = F - neucoeff * Fneu
# baseline operation
Fc = dcnv.preprocess(
     F=Fc,
     baseline='maximin',
     win_baseline=60.0, # sec
     sig_baseline=10,   # frames
     fs=fs,
     prctile_baseline=8.0, # percentile of trace to use as baseline
     batch_size=batch_size,
     device=torch.device('cuda')
 )

In [6]:
print(Fc.shape, Fc.min(), Fc.max())

(746, 54000) -174.97864 1896.7598


### Motion energy

In [7]:
# motion energy data
for session in all_session_dir:
    motion = np.load(os.path.join(session, 'move_deve', 'motion_energy_glob.npy'))
    print('shape', motion.shape, 'min:', '%.2f' % motion.min(), 'max:', '%.2f' % motion.max())

shape (54000,) min: 0.00 max: 36783720.00
shape (54000,) min: 0.00 max: 41070356.00
shape (54000,) min: 0.00 max: 46979495.00
shape (54000,) min: 0.00 max: 49539019.00
shape (53999,) min: 0.00 max: 43031877.00
shape (54000,) min: 0.00 max: 50919792.00
shape (54000,) min: 0.00 max: 47275140.00


In [8]:
# figure out where the frame drop happened
dt = np.load(os.path.join(all_session_dir[4], 'move_deve', 'interframe_int.npy')) * 1000
np.where(dt > 0.04)

(array([22977]),)

### Data reformat

In [ ]:
neucoeff = 0.7
fs = 30
batch_size = 128
device = torch.device('cuda')

all_Fc = []
all_me = []

print('Processing data for %s' % subject)

for session in all_session_dir:
    session_name = os.path.basename(session)
    print(f'--- {session_name} ---')

    # ---- calcium preprocessing ----
    F = np.load(os.path.join(session, 'suite2p', 'plane0', 'F.npy'))
    Fneu = np.load(os.path.join(session, 'suite2p', 'plane0', 'Fneu.npy'))

    Fc = F - neucoeff * Fneu
    Fc = dcnv.preprocess(
        F=Fc,
        baseline='maximin',
        win_baseline=60.0,
        sig_baseline=10,
        fs=fs,
        prctile_baseline=8.0,
        batch_size=batch_size,
        device=device,
    )
    all_Fc.append(Fc)
    expected_len = Fc.shape[1]
    print(f'  Fc: {Fc.shape}  range [{Fc.min():.2f}, {Fc.max():.2f}]  mean={Fc.mean():.2f}')

    # ---- motion energy: interpolate dropped frames, then normalize ----
    me = np.load(os.path.join(session, 'move_deve', 'motion_energy_glob.npy'))
    dt = np.load(os.path.join(session, 'move_deve', 'interframe_int.npy'))

    if me.shape[0] < expected_len:
        drop_indices = np.where(dt * 1000 > 0.04)[0]
        print(f'  frame drops at indices: {drop_indices}  ({len(drop_indices)} drops)')
        for offset, idx in enumerate(drop_indices):
            insert_pos = idx + 1 + offset
            interp_val = (me[insert_pos - 1] + me[insert_pos]) / 2.0
            me = np.insert(me, insert_pos, interp_val)
        print(f'  after interpolation: {me.shape}')

    assert me.shape[0] == expected_len, f'unexpected length {me.shape[0]} vs {expected_len}'

    me_norm = me / me.std()
    all_me.append(me_norm)
    print(f'  motion energy: range [{me_norm.min():.4f}, {me_norm.max():.4f}]  mean={me_norm.mean():.4f}  std={me_norm.std():.4f}')

print(f'\n=== summary ===')
print(f'sessions: {len(all_Fc)}')
print(f'neurons: {all_Fc[0].shape[0]},  frames per session: {all_Fc[0].shape[1]}')
print(f'FC shapes:  {[fc.shape for fc in all_Fc]}')
print(f'ME shapes:  {[me.shape for me in all_me]}')


Processing data for jm039
--- 2024-04-30_a ---
  Fc: (746, 54000)  range [-174.98, 1896.76]  mean=15.24
  motion energy: range [0.0000, 18.5727]  mean=0.7518  std=1.0000
--- 2024-05-01_a ---
  Fc: (746, 54000)  range [-226.38, 1811.42]  mean=17.35
  motion energy: range [0.0000, 17.1148]  mean=0.6328  std=1.0000
--- 2024-05-02_a ---
  Fc: (746, 54000)  range [-291.58, 2241.40]  mean=20.09
  motion energy: range [0.0000, 19.1041]  mean=0.6324  std=1.0000
--- 2024-05-03_a ---
  Fc: (746, 54000)  range [-273.11, 2408.16]  mean=19.03
  motion energy: range [0.0000, 17.5883]  mean=0.5882  std=1.0000
--- 2024-05-04_a ---
  Fc: (746, 54000)  range [-229.77, 2454.48]  mean=23.15
  frame drops at indices: [22977]  (1 drops)
  after interpolation: (54000,)
  motion energy: range [0.0000, 16.3035]  mean=0.6144  std=1.0000
--- 2024-05-05_a ---
  Fc: (746, 54000)  range [-441.73, 2925.55]  mean=23.82
  motion energy: range [0.0000, 18.0875]  mean=0.5645  std=1.0000
--- 2024-05-06_a ---
  Fc: (746, 